1. ETF Core Data
2. Financial News & Events data
3. Macro-Economic Indicators
4. Risk-Free Rate Data
5. Historical Crisis/Event Data for Scenario Testing
6. User Risk Profiling Reference Data


In [1]:
# pip install etfpy

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf


In [3]:
from etfpy import ETF, load_etf, get_available_etfs_list

# returns list of available ETFs.
etfs = get_available_etfs_list()
print(f"Available ETFs: {etfs[:10]}... \nTotal: {len(etfs)}")

Available ETFs: ['SPY', 'IVV', 'VOO', 'VTI', 'QQQ', 'VEA', 'VUG', 'VTV', 'IEFA', 'BND']... 
Total: 3262


In [4]:
# spy = load_etf('SPY')
# spy.get_quotes(interval="daily", periods=10)

In [1]:
import os
import time
from pathlib import Path
import warnings

# Suppress all warnings and debug output
warnings.filterwarnings('ignore')
import logging
logging.getLogger('yfinance').setLevel(logging.ERROR)

# Create directories if they don't exist
# os.makedirs('project/data/raw/prices', exist_ok=True)
# os.makedirs('project/data/processed', exist_ok=True)

# Define date range
start_date = '1999-01-01'  # 10 years of data
end_date = '2025-10-01'

# Initialize list to store all data
all_data = []
failed_tickers = []

print(f"Starting download for {len(etfs)} ETFs...\n")
# data = yf.download('SPY', start=start_date, end=end_date, progress=True)

# Loop through all ETFs and download data
for idx, etf in enumerate(etfs, 1):
    ticker = etf.ticker if hasattr(etf, 'ticker') else str(etf)
    try:
        print(f"[{idx}/{len(etfs)}] Downloading {ticker}...", end=" ")
        
        # Download data
        data = yf.download(ticker, start=start_date, end=end_date, progress=False)
        
        if data is not None and len(data) > 0:
            # Add ticker column
            data['Ticker'] = ticker
            data = data.reset_index()
            
            # Save individual CSV
            data.to_csv(f'project/data/raw/prices/{ticker}.csv', index=False)
            
            # Append to combined list
            all_data.append(data)
            
            print(f"✓ ({len(data)} records)")
        else:
            print(f"✗ No data")
            failed_tickers.append(ticker)
            
    except Exception as e:
        print(f"✗ Error: {str(e)[:50]}")
        failed_tickers.append(ticker)

    # Rate limiting: 2 second delay between requests
    time.sleep(2)

NameError: name 'sg_etfs_list' is not defined

In [9]:
import pandas as pd
import os
from datetime import datetime, timedelta
import glob

def analyze_etf_data_freshness(data_dir='/Users/linhtrankhanh/Downloads/fyp/project/data/raw/prices'):
    """
    Analyze all ETF CSV files to check data freshness and identify inactive ETFs
    """
    print("="*80)
    print("ETF DATA FRESHNESS ANALYSIS")
    print("="*80)
    
    # Get all CSV files in the directory
    csv_files = glob.glob(os.path.join(data_dir, "*.csv"))
    
    if not csv_files:
        print(f"❌ No CSV files found in {data_dir}")
        return None
    
    print(f"Found {len(csv_files)} CSV files to analyze...\n")
    
    # Analysis results
    results = []
    today = datetime.now()
    cutoff_date = today - timedelta(days=30)  # Consider stale if older than 30 days
    
    for csv_file in csv_files:
        try:
            # Extract ticker from filename
            ticker = os.path.basename(csv_file).replace('.csv', '')
            
            # Read CSV file
            df = pd.read_csv(csv_file)
            
            if df.empty:
                results.append({
                    'ticker': ticker,
                    'status': 'EMPTY',
                    'records': 0,
                    'latest_date': None,
                    'earliest_date': None,
                    'days_since_update': None,
                    'file_size_kb': round(os.path.getsize(csv_file) / 1024, 2)
                })
                continue
            
            # Convert Date column to datetime
            df['Date'] = pd.to_datetime(df['Date'])
            
            # Get date range
            latest_date = df['Date'].max()
            earliest_date = df['Date'].min()
            days_since_update = (today - latest_date).days
            
            # Determine status
            if days_since_update <= 7:
                status = 'ACTIVE'
            elif days_since_update <= 30:
                status = 'RECENT'
            elif days_since_update <= 90:
                status = 'STALE'
            else:
                status = 'INACTIVE'
            
            results.append({
                'ticker': ticker,
                'status': status,
                'records': len(df),
                'latest_date': latest_date.strftime('%Y-%m-%d'),
                'earliest_date': earliest_date.strftime('%Y-%m-%d'),
                'days_since_update': days_since_update,
                'file_size_kb': round(os.path.getsize(csv_file) / 1024, 2)
            })
            
        except Exception as e:
            results.append({
                'ticker': ticker,
                'status': 'ERROR',
                'records': 0,
                'latest_date': None,
                'earliest_date': None,
                'days_since_update': None,
                'file_size_kb': round(os.path.getsize(csv_file) / 1024, 2),
                'error': str(e)[:50]
            })
    
    # Convert to DataFrame for analysis
    results_df = pd.DataFrame(results)
    
    # Sort by status and days since update
    status_order = {'ACTIVE': 1, 'RECENT': 2, 'STALE': 3, 'INACTIVE': 4, 'EMPTY': 5, 'ERROR': 6}
    results_df['sort_order'] = results_df['status'].map(status_order)
    results_df = results_df.sort_values(['sort_order', 'days_since_update'])
    
    # Print summary statistics
    print("SUMMARY STATISTICS:")
    print("-" * 50)
    status_counts = results_df['status'].value_counts()
    for status, count in status_counts.items():
        print(f"{status:12s}: {count:4d} ETFs")
    
    print(f"\nTotal ETFs analyzed: {len(results_df)}")
    
    # Show detailed results
    print("\n" + "="*80)
    print("DETAILED RESULTS:")
    print("="*80)
    
    # Display results by category
    for status in ['ACTIVE', 'RECENT', 'STALE', 'INACTIVE', 'EMPTY', 'ERROR']:
        subset = results_df[results_df['status'] == status]
        if len(subset) > 0:
            print(f"\n🔍 {status} ETFs ({len(subset)}):")
            print("-" * 50)
            
            if status == 'ACTIVE':
                # Show only first 10 active ETFs
                display_df = subset.head(10)[['ticker', 'latest_date', 'records', 'days_since_update']]
                print(display_df.to_string(index=False))
                if len(subset) > 10:
                    print(f"... and {len(subset) - 10} more active ETFs")
            
            elif status in ['INACTIVE', 'STALE']:
                # Show all inactive/stale ETFs
                display_df = subset[['ticker', 'latest_date', 'records', 'days_since_update']]
                print(display_df.to_string(index=False))
                
            elif status == 'ERROR':
                # Show error ETFs with error message
                display_df = subset[['ticker', 'file_size_kb', 'error']]
                print(display_df.to_string(index=False))
                
            else:
                # Show other categories
                display_df = subset[['ticker', 'latest_date', 'records', 'days_since_update']]
                print(display_df.to_string(index=False))
    
    # Save results to CSV
    output_file = '/Users/linhtrankhanh/Downloads/fyp/project/data/processed/etf_data_analysis.csv'
    results_df.drop('sort_order', axis=1).to_csv(output_file, index=False)
    
    print(f"\n📊 Full analysis saved to: {output_file}")
    
    # Recommendations
    print("\n" + "="*80)
    print("RECOMMENDATIONS:")
    print("="*80)
    
    active_count = len(results_df[results_df['status'] == 'ACTIVE'])
    inactive_count = len(results_df[results_df['status'] == 'INACTIVE'])
    
    print(f"✅ Keep {active_count} ACTIVE ETFs for your recommendation engine")
    print(f"❌ Remove {inactive_count} INACTIVE ETFs (no recent data)")
    
    if len(results_df[results_df['status'] == 'STALE']) > 0:
        print(f"⚠️  Review {len(results_df[results_df['status'] == 'STALE'])} STALE ETFs (may be delisted)")
    
    # Return active ticker list
    active_tickers = results_df[results_df['status'].isin(['ACTIVE', 'RECENT'])]['ticker'].tolist()
    print(f"\n🎯 {len(active_tickers)} ETFs recommended for your advisor:")
    print(f"   {', '.join(active_tickers[:20])}")
    if len(active_tickers) > 20:
        print(f"   ... and {len(active_tickers) - 20} more")
    
    return results_df

# Run the analysis
analysis_results = analyze_etf_data_freshness()

ETF DATA FRESHNESS ANALYSIS
Found 3259 CSV files to analyze...

SUMMARY STATISTICS:
--------------------------------------------------
RECENT      : 3061 ETFs
INACTIVE    :  170 ETFs
STALE       :   28 ETFs

Total ETFs analyzed: 3259

DETAILED RESULTS:

🔍 RECENT ETFs (3061):
--------------------------------------------------
ticker latest_date  records  days_since_update
  DSCF  2025-09-30     1013                 22
  GOOY  2025-09-30      547                 22
   LCF  2025-09-30      783                 22
  VCAR  2025-09-30     1196                 22
  GABF  2025-09-30      852                 22
  SPXU  2025-09-30     4093                 22
   PPA  2025-09-30     5014                 22
  USTB  2025-09-30     1992                 22
  XAUG  2025-09-30      531                 22
  XSHD  2025-09-30     2215                 22
   BCI  2025-09-30     2138                 22
  BBRE  2025-09-30     1833                 22
  TRND  2025-09-30     1612                 22
  QDIV  2025-09

In [11]:
def filter_quality_etfs(analysis_results):
    """
    Filter ETFs suitable for recommendation engine based on quality criteria
    """
    print("="*80)
    print("QUALITY ETF FILTERING")
    print("="*80)
    
    # Define quality criteria
    quality_etfs = analysis_results[
        (analysis_results['status'].isin(['ACTIVE', 'RECENT'])) &  # Currently trading
        (analysis_results['records'] >= 1000) &  # At least ~4 years of data (250 trading days/year)
        (analysis_results['days_since_update'] <= 30) &  # Recent data
        (analysis_results['file_size_kb'] > 50)  # Substantial data (not tiny files)
    ].copy()
    
    print(f"📊 Filtering Results:")
    print(f"   Total ETFs analyzed: {len(analysis_results)}")
    print(f"   Quality ETFs found: {len(quality_etfs)}")
    print(f"   Filtered out: {len(analysis_results) - len(quality_etfs)}")
    
    # Show quality ETFs sorted by data length
    quality_etfs_sorted = quality_etfs.sort_values('records', ascending=False)
    
    print(f"\n✅ TOP QUALITY ETFs (sorted by data length):")
    print("-" * 60)
    display_cols = ['ticker', 'latest_date', 'records', 'days_since_update']
    print(quality_etfs_sorted[display_cols].head(20).to_string(index=False))
    
    # Save quality ETF list
    quality_tickers = quality_etfs_sorted['ticker'].tolist()
    
    # Export for your recommendation engine
    quality_df = quality_etfs_sorted[['ticker', 'records', 'latest_date', 'earliest_date']]
    output_file = '/Users/linhtrankhanh/Downloads/fyp/project/data/processed/quality_etfs.csv'
    quality_df.to_csv(output_file, index=False)
    
    print(f"\n💾 Quality ETF list saved to: {output_file}")
    print(f"\n🎯 Recommended ETFs for your advisor: {len(quality_tickers)}")
    
    return quality_tickers, quality_etfs_sorted

# Apply quality filtering
if 'analysis_results' in locals():
    quality_tickers, quality_etfs = filter_quality_etfs(analysis_results)
else:
    print("❌ Run the analysis first!")

QUALITY ETF FILTERING
📊 Filtering Results:
   Total ETFs analyzed: 3259
   Quality ETFs found: 2202
   Filtered out: 1057

✅ TOP QUALITY ETFs (sorted by data length):
------------------------------------------------------------
ticker latest_date  records  days_since_update
   EWM  2025-09-30     6728                 22
   XLU  2025-09-30     6728                 22
   EWL  2025-09-30     6728                 22
   EWQ  2025-09-30     6728                 22
   EWI  2025-09-30     6728                 22
   EWO  2025-09-30     6728                 22
   EWG  2025-09-30     6728                 22
   XLV  2025-09-30     6728                 22
   EWU  2025-09-30     6728                 22
   EWD  2025-09-30     6728                 22
   EWS  2025-09-30     6728                 22
   EWK  2025-09-30     6728                 22
   XLY  2025-09-30     6728                 22
   XLP  2025-09-30     6728                 22
   EWW  2025-09-30     6728                 22
   XLK  2025-09-30  

In [14]:
# def validate_etf_quality_improved(ticker_list):
#     """
#     Improved ETF quality validation with realistic thresholds
#     """
#     print(f"\n🔍 Validating {len(ticker_list)} ETFs with IMPROVED criteria...")
    
#     validated_etfs = []
    
#     for idx, ticker in enumerate(ticker_list, 1):
#         try:
#             print(f"[{idx}/{len(ticker_list)}] Processing {ticker}...", end=" ")
            
#             etf = yf.Ticker(ticker)
#             info = etf.info
            
#             # Quality criteria
#             aum = info.get('totalAssets', 0)
#             volume = info.get('averageVolume', 0)
#             inception = info.get('fundInceptionDate')
            
#             # Calculate fund age
#             if inception:
#                 from datetime import datetime
#                 inception_date = datetime.fromtimestamp(inception)
#                 age_years = (datetime.now() - inception_date).days / 365.25
#             else:
#                 age_years = 0
            
#             # IMPROVED Quality scoring
#             quality_score = 0
            
#             # AUM scoring
#             if aum > 1_000_000_000:    # $1B+ (mega ETFs)
#                 quality_score += 3
#             elif aum > 500_000_000:    # $500M+ (large ETFs)
#                 quality_score += 2
#             elif aum > 100_000_000:    # $100M+ (medium ETFs)
#                 quality_score += 1
            
#             # REALISTIC Volume scoring
#             if volume > 500_000:       # Very liquid
#                 quality_score += 3
#             elif volume > 100_000:     # Liquid
#                 quality_score += 2
#             elif volume > 25_000:      # ✅ ADEQUATE (lowered from 100K)
#                 quality_score += 1
#             # 10K-25K still acceptable for smaller investors
            
#             # Age scoring
#             if age_years > 10:         # Very established
#                 quality_score += 3
#             elif age_years > 5:        # Established
#                 quality_score += 2
#             elif age_years > 3:        # Mature enough
#                 quality_score += 1
            
#             validated_etfs.append({
#                 'ticker': ticker,
#                 'name': info.get('shortName', ''),
#                 'aum': aum,
#                 'volume': volume,
#                 'age_years': round(age_years, 1),
#                 'quality_score': quality_score
#             })
            
#             print(f"✓ Score {quality_score}/9")
            
#         except Exception as e:
#             print(f"✗ Error: {str(e)[:30]}")
            
#         time.sleep(0.5)  # Faster rate limiting
    
#     # Sort and categorize
#     validated_df = pd.DataFrame(validated_etfs)
#     validated_df = validated_df.sort_values('quality_score', ascending=False)
    
#     # New quality tiers (out of 9)
#     excellent = validated_df[validated_df['quality_score'] >= 7]['ticker'].tolist()  # 7-9/9
#     very_good = validated_df[validated_df['quality_score'] >= 5]['ticker'].tolist()  # 5-6/9
#     good = validated_df[validated_df['quality_score'] >= 3]['ticker'].tolist()       # 3-4/9
    
#     print(f"\n🏆 IMPROVED QUALITY TIERS:")
#     print(f"   EXCELLENT (7-9/9): {len(excellent)} ETFs")
#     print(f"   VERY GOOD (5-6/9): {len(very_good)} ETFs") 
#     print(f"   GOOD (3-4/9): {len(good)} ETFs")
    
#     print(f"\n🎯 Recommended for your advisor:")
#     print(f"   Use VERY GOOD+ ETFs: {len(very_good)} total")
    
#     return very_good, validated_df

# # Use improved validation
# if 'quality_tickers' in locals():
#     improved_etfs, improved_validation = validate_etf_quality_improved(quality_tickers)
#     print(f"\n✅ IMPROVED recommendation: Use {len(improved_etfs)} quality ETFs!")


🔍 Validating 2202 ETFs with IMPROVED criteria...
[1/2202] Processing EWM... ✓ Score 6/9
[2/2202] Processing XLU... ✓ Score 9/9
[3/2202] Processing EWL... ✓ Score 8/9
[4/2202] Processing EWQ... ✓ Score 6/9
[5/2202] Processing EWI... ✓ Score 7/9
[6/2202] Processing EWO... ✓ Score 5/9
[7/2202] Processing EWG... ✓ Score 9/9
[8/2202] Processing XLV... ✓ Score 9/9
[9/2202] Processing EWU... ✓ Score 9/9
[10/2202] Processing EWD... ✓ Score 5/9
[11/2202] Processing EWS... ✓ Score 8/9
[12/2202] Processing EWK... ✓ Score 4/9
[13/2202] Processing XLY... ✓ Score 9/9
[14/2202] Processing XLP... ✓ Score 9/9
[15/2202] Processing EWW... ✓ Score 9/9
[16/2202] Processing XLK... ✓ Score 9/9
[17/2202] Processing EWC... ✓ Score 9/9
[18/2202] Processing XLF... ✓ Score 9/9
[19/2202] Processing XLI... ✓ Score 9/9
[20/2202] Processing EWJ... ✓ Score 9/9
[21/2202] Processing EWA... ✓ Score 9/9
[22/2202] Processing XLB... ✓ Score 9/9
[23/2202] Processing XLE... ✓ Score 9/9
[24/2202] Processing EWP... ✓ Score 8/9

In [ ]:
def collect_comprehensive_etf_metadata(ticker_list):
    """
    Collect ALL available metadata fields from yfinance .info for each ETF
    Dynamically adds new columns as new fields are discovered
    """
    print("="*80)
    print("COMPREHENSIVE ETF METADATA COLLECTION")
    print("="*80)
    print(f"🔄 Processing {len(ticker_list)} ETFs...")
    print("⏳ This will take several minutes with rate limiting...\n")
    
    all_metadata = []
    all_columns_seen = set()
    errors = []
    
    for idx, ticker in enumerate(ticker_list, 1):
        try:
            print(f"[{idx:3d}/{len(ticker_list)}] {ticker:6s}", end=" ", flush=True)
            
            # Get ETF info from yfinance
            etf = yf.Ticker(ticker)
            info = etf.info
            
            if not info:
                print("❌ No info available")
                errors.append(f"{ticker}: No info returned")
                continue
            
            # Start with ticker as primary key
            metadata_row = {'ticker': ticker}
            
            # Add ALL fields from .info
            for key, value in info.items():
                # Clean up the field name
                clean_key = str(key).lower().replace(' ', '_').replace('-', '_')
                metadata_row[clean_key] = value
                all_columns_seen.add(clean_key)
            
            all_metadata.append(metadata_row)
            print(f"✓ ({len(info)} fields collected)")
            
        except Exception as e:
            error_msg = f"{ticker}: {str(e)[:40]}"
            errors.append(error_msg)
            print(f"❌ Error: {str(e)[:30]}")
        
        # Rate limiting to avoid getting blocked
        time.sleep(1.0)
    
    # Convert to DataFrame
    if all_metadata:
        print(f"\n📊 Creating comprehensive metadata DataFrame...")
        metadata_df = pd.DataFrame(all_metadata)
        
        # Fill missing values appropriately
        for col in metadata_df.columns:
            if col != 'ticker':
                # Fill NaN with appropriate default based on data type
                if metadata_df[col].dtype == 'object':
                    metadata_df[col] = metadata_df[col].fillna('')
                else:
                    metadata_df[col] = metadata_df[col].fillna(0)
        
        # Sort columns logically
        important_cols = ['ticker', 'shortname', 'longname', 'symbol', 'quotetype', 'currency', 
                         'exchange', 'category', 'fundfamily', 'legaltype', 'region',
                         'totalassets', 'averagevolume', 'regularmarketprice', 'navprice',
                         'yield', 'ytdreturn', 'threeyearaveragereturn', 'fiveyearaveragereturn',
                         'beta3year', 'fiftytwoweekhigh', 'fiftytwoweeklow', 'fundinceptiondate']
        
        # Reorder columns: important first, then alphabetical for the rest
        available_important = [col for col in important_cols if col in metadata_df.columns]
        other_cols = sorted([col for col in metadata_df.columns if col not in available_important])
        ordered_cols = available_important + other_cols
        
        metadata_df = metadata_df[ordered_cols]
        
        # Save to CSV
        output_file = '/Users/linhtrankhanh/Downloads/fyp/project/data/processed/comprehensive_etf_metadata.csv'
        os.makedirs('/Users/linhtrankhanh/Downloads/fyp/project/data/processed', exist_ok=True)
        metadata_df.to_csv(output_file, index=False)
        
        print(f"\n✅ COLLECTION COMPLETE!")
        print(f"{'='*80}")
        print(f"📈 Successfully collected: {len(metadata_df)} ETFs")
        print(f"📊 Total data fields: {len(metadata_df.columns)} columns")
        print(f"💾 Saved to: {output_file}")
        
        if errors:
            print(f"⚠️  Errors encountered: {len(errors)}")
            print(f"   Failed tickers: {[e.split(':')[0] for e in errors]}")
        
        # Show sample of the data
        print(f"\n📋 SAMPLE DATA (first 5 ETFs, key columns):")
        sample_cols = ['ticker', 'shortname', 'category', 'totalassets', 'averagevolume', 'currency']
        available_sample_cols = [col for col in sample_cols if col in metadata_df.columns]
        print(metadata_df[available_sample_cols].head().to_string(index=False))
        
        # Show column breakdown
        print(f"\n📑 COLUMN CATEGORIES:")
        identification_cols = [col for col in metadata_df.columns if any(word in col for word in ['name', 'symbol', 'ticker', 'type', 'category', 'family'])]
        financial_cols = [col for col in metadata_df.columns if any(word in col for word in ['price', 'return', 'yield', 'assets', 'volume', 'beta', 'ratio'])]
        date_cols = [col for col in metadata_df.columns if any(word in col for word in ['date', 'time', 'inception'])]
        
        print(f"   🏷️  Identification ({len(identification_cols)}): {', '.join(identification_cols[:5])}...")
        print(f"   💰 Financial ({len(financial_cols)}): {', '.join(financial_cols[:5])}...")
        print(f"   📅 Date/Time ({len(date_cols)}): {', '.join(date_cols[:3])}...")
        print(f"   📊 Other ({len(metadata_df.columns) - len(identification_cols) - len(financial_cols) - len(date_cols)})")
        
        return metadata_df, errors
    
    else:
        print("❌ No metadata collected!")
        return None, errors

# Run comprehensive metadata collection
if 'quality_tickers' in locals() and len(quality_tickers) > 0:
    print(f"🎯 Using {len(quality_tickers)} quality ETFs from previous analysis")
    comprehensive_metadata, collection_errors = collect_comprehensive_etf_metadata(quality_tickers)
else:
    print("❌ No quality_tickers found! Run the quality filtering first.")